In [11]:
import sys

print(sys.executable)
print(sys.version)
print("\nPython Paths:\n")
for path in sys.path:
    print(path)
print("\n")

!{sys.executable} -m pip install -U python-woc pandas matplotlib

/usr/bin/python3.11
3.11.13 (main, Jun  1 2026, 00:00:00) [GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]

Python Paths:

/usr/lib64/python311.zip
/usr/lib64/python3.11
/usr/lib64/python3.11/lib-dynload

/home/cdamron3/.local/lib/python3.11/site-packages
/usr/lib64/python3.11/site-packages
/usr/lib/python3.11/site-packages


Defaulting to user installation because normal site-packages is not writeable


In [16]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'MariaNattestad/Ribbon',
    'stan-dev/cmdstanpy',
    'rveltz/PiecewiseDeterministicMarkovProcesses.jl',
    'lpwgroup/crank',
    'thelfer/MFrontGenericInterfaceSupport',
    'jonescompneurolab/hnn',
    'mirnylab/cooler',
    'jhuapl-boss/intern',
    'privefl/bigstatsr',
    'xeokit/xeokit-sdk'
]

list_df_commits = []

for prj in projects:
    # Convert GitHub repository names to WoC V2412 project names
    woc_prj = prj.lower().replace('/','_',2)

    print(f"Getting commits for {woc_prj}...")

    try:
        commits = woc.get_values('p2c', woc_prj)
        df = pd.DataFrame(commits, columns=["sha1"])
        df['project'] = woc_prj
        list_df_commits.append(df)

        print(f"  Found {len(df)} commits")
    except KeyError as e:
        print(f"  ERROR retrieving {woc_prj}: {e}")
        

df = pd.concat(list_df_commits, ignore_index = True)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv', index = False)
df.head(1)

Getting commits for marianattestad_ribbon...
  Found 519 commits
Getting commits for stan-dev_cmdstanpy...
  Found 2742 commits
Getting commits for rveltz_piecewisedeterministicmarkovprocesses.jl...
  Found 608 commits
Getting commits for lpwgroup_crank...
  Found 370 commits
Getting commits for thelfer_mfrontgenericinterfacesupport...
  Found 753 commits
Getting commits for jonescompneurolab_hnn...
  Found 2050 commits
Getting commits for mirnylab_cooler...
  Found 1406 commits
Getting commits for jhuapl-boss_intern...
  Found 1475 commits
Getting commits for privefl_bigstatsr...
  Found 1007 commits
Getting commits for xeokit_xeokit-sdk...
  Found 15141 commits


,sha1,project
0,0023779e992f3962eecf62e652ba81fe66da3852,marianattestad_ribbon


In [17]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
from tqdm import tqdm

# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())
  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch', chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 17%|█████████████▌                                                                 | 446/2608 [07:43<38:02,  1.06s/it]

Got Errors {'3e201a4346497f954f6feb069ce688c6e6bd299e': 'Key 3e201a4346497f954f6feb069ce688c6e6bd299e not found in /da5_fast/All.sha1c/commit_62.tch'}


 17%|█████████████▊                                                                 | 456/2608 [07:54<37:31,  1.05s/it]

Got Errors {'5fda28b204b7644c981ba18ca3cfedf55667b709': 'Key 5fda28b204b7644c981ba18ca3cfedf55667b709 not found in /da5_fast/All.sha1c/commit_95.tch'}


 20%|███████████████▌                                                               | 513/2608 [08:53<36:21,  1.04s/it]

...

Got Errors {'fd92a8d9c4e7bc2a5d71a9f480fc22bb8b15d156': 'Key fd92a8d9c4e7bc2a5d71a9f480fc22bb8b15d156 not found in /da5_fast/All.sha1c/commit_125.tch'}


100%|█████████████████████████████████████████████████████████████████████████████▉| 2604/2608 [45:13<00:04,  1.04s/it]

Got Errors {'ff66e348fc6786a85f8ff8732f9876d63e3782d6': 'Key ff66e348fc6786a85f8ff8732f9876d63e3782d6 not found in /da5_fast/All.sha1c/commit_127.tch'}


100%|██████████████████████████████████████████████████████████████████████████████| 2608/2608 [45:17<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0023779e992f3962eecf62e652ba81fe66da3852,5b8285fac196b42042d3cdc10490502feb433a0f,[849cfe159470e7600570cac179b8e69e2edc3e64],Robert Aboukhalil <robert.aboukhalil@gmail.com>,1584246680,-0700,Robert Aboukhalil <robert.aboukhalil@gmail.com>,1584246680,-0700,Small fix\n,0023779e992f3962eecf62e652ba81fe66da3852,marianattestad_ribbon
1,00c7ec6d1edf1ee4fef4e87b9e8254271f76fd2e,71c0c4a2afed052ec0621457b27f6dfa0cd5800d,"[240b6a5c8cea34909572fa8fd0c5d585dc0d5a78, a22...",Maria Nattestad <maria.nattestad@gmail.com>,1472156867,-0700,Maria Nattestad <maria.nattestad@gmail.com>,1472156867,-0700,Merge branch 'master' into web-production\n,00c7ec6d1edf1ee4fef4e87b9e8254271f76fd2e,marianattestad_ribbon


In [37]:
print("Original Number of Commits: ", len(df))
print("Recieved Number of Commits: ", len(df_commit_data))
print("Missing Number of Commits: ", len(df) - len(df_commit_data))

Original Number of Commits:  26071
Recieved Number of Commits:  25776
Missing Number of Commits:  295


In [38]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0023779e992f3962eecf62e652ba81fe66da3852', 'tree': '5b8285fac196b42042d3cdc10490502feb433a0f', 'parent': ['849cfe159470e7600570cac179b8e69e2edc3e64'], 'author': 'Robert Aboukhalil <robert.aboukhalil@gmail.com>', 'author_time': 1584246680, 'author_tz': '-0700', 'committer': 'Robert Aboukhalil <robert.aboukhalil@gmail.com>', 'committer_time': 1584246680, 'committer_tz': '-0700', 'message': 'Small fix\n'}


In [44]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()
dfinf = dfinf.rename(columns={
    'project': 'project_wocid',
    'commit': 'commit_sha1',
    'author_time': 'time',
    'message': 'commit_message'
})

print(dfinf['project_wocid'].value_counts())
print('\n')

dfinf.head(1)


project_wocid
xeokit_xeokit-sdk                                  14856
stan-dev_cmdstanpy                                  2742
jonescompneurolab_hnn                               2049
jhuapl-boss_intern                                  1471
mirnylab_cooler                                     1403
privefl_bigstatsr                                   1007
thelfer_mfrontgenericinterfacesupport                751
rveltz_piecewisedeterministicmarkovprocesses.jl      608
marianattestad_ribbon                                519
lpwgroup_crank                                       370
Name: count, dtype: int64




,project_wocid,commit_sha1,author,time,commit_message
0,marianattestad_ribbon,0023779e992f3962eecf62e652ba81fe66da3852,Robert Aboukhalil <robert.aboukhalil@gmail.com>,1584246680,Small fix\n


In [45]:
#mode a means append, so you have all your projects in the same file
yournetid='cdamron3'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False, sep=';')

In [48]:
woc_summary = dfinf.groupby('project_wocid').agg(
    commits=('commit_sha1', 'count'),
    authors=('author', 'nunique'),
    max_time=('time', 'max'),
    min_time=('time', 'min')
).reset_index()

woc_summary

,project_wocid,commits,authors,max_time,min_time
0,jhuapl-boss_intern,1471,45,1745859342,1443622463
1,jonescompneurolab_hnn,2049,39,1761687929,1487864695
2,lpwgroup_crank,370,27,1746254838,1520143708
3,marianattestad_ribbon,519,7,1751099438,1467246668
4,mirnylab_cooler,1403,47,1762187309,1447703659
5,privefl_bigstatsr,1007,13,1753781436,1473922251
6,rveltz_piecewisedeterministicmarkovprocesses.jl,608,14,1772783699,1450270615
7,stan-dev_cmdstanpy,2742,83,1762364056,1546473710
8,thelfer_mfrontgenericinterfacesupport,751,25,1762378091,1529498553
9,xeokit_xeokit-sdk,14856,116,1762378885,1543685191


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github